# Checks on the feature template (optional)

This notebook is **not needed to build the features**. It re-runs `01_feature_pipeline_template.ipynb` and then verifies it:

1. Data-quality checks on the raw tables
2. Cross-checks of the calendar and market series against the project repo
3. A look-ahead test: no feature for day *t* uses data from day *t*
4. The feature row a live forecast would use for the next trading day
5. The E0 → E5 results compared with the midterm slide

In [1]:
# Run every code cell of the template notebook here, so its tables and functions are available below
import json
template = json.load(open("01_feature_pipeline_template.ipynb"))
for cell in template["cells"]:
    if cell["cell_type"] == "code":
        exec("".join(cell["source"]))
print(f"Template run: {len(features):,} stock-days, {len(FEATURE_COLUMNS)} features")

Data folder: /Users/kunj/Desktop/blackrock_auction_vol/blackrock_data


CRSP daily    :  4,927,078 rows  (2011-01-03 to 2025-12-31)
Auction daily :  3,594,398 rows  (2015-01-02 to 2025-12-31)
Options daily :  3,488,919 rows  (2015-01-02 to 2025-08-29)


3,594,404 stock-days, 40 features, 3,466,653 model-ready rows


Saved 3,594,404 rows to outputs/model_features
Template run: 3,594,404 stock-days, 40 features


In [2]:
import time
QC = []
def check(name, passed, detail="", warn_only=False):
    status = "PASS" if bool(passed) else ("WARN" if warn_only else "FAIL")
    QC.append({"check": name, "status": status, "detail": str(detail)})
    print(f"[{status}] {name}" + (f" — {detail}" if detail != "" else ""))

def read_file(path):
    out = duckdb.connect().execute(f"SELECT * FROM read_parquet('{path}')").df()
    out["date"] = pd.to_datetime(out["date"]).astype("datetime64[ns]")
    return out

## 1. Raw data

In [3]:
check("CRSP (date, permno) unique", not crsp_raw.duplicated(["date", "permno"]).any())
check("Auction (date, permno) unique", not auction_raw.duplicated(["date", "permno"]).any())
check("CRSP daily volume non-negative", (crsp_raw["daily_volume"].dropna() >= 0).all())
check("Auction volume positive where present", (auction_raw["auction_volume"].dropna() > 0).all())
check("target_found agrees with auction_volume presence", (auction_raw["target_found"].eq(True) == auction_raw["auction_volume"].notna()).all())
both = full["observed_target"] & (full["daily_volume"] > 0)
check("Auction volume never exceeds CRSP daily volume", (full.loc[both, "auction_volume"] <= full.loc[both, "daily_volume"]).all())
unmatched = len(auction_raw) - auction_raw.merge(crsp_raw[["date", "permno"]], on=["date", "permno"]).shape[0]
check("Every auction row matches a CRSP stock-day", unmatched == 0, f"{unmatched} unmatched")
if options_raw is not None:
    check("Options (date, permno) unique", not options_raw.duplicated(["date", "permno"]).any())
    check("Options volume / open interest non-negative", (options_raw[["opt_total_oi", "opt_total_vol", "oi_exp_today", "oi_exp_7d"]].fillna(0) >= 0).all().all())
    check("Open interest expiring within 7 days never exceeds total", (options_raw["oi_exp_7d"].fillna(0) <= options_raw["opt_total_oi"].fillna(0) + 1e-9).all())

[PASS] CRSP (date, permno) unique


[PASS] Auction (date, permno) unique
[PASS] CRSP daily volume non-negative
[PASS] Auction volume positive where present
[PASS] target_found agrees with auction_volume presence
[PASS] Auction volume never exceeds CRSP daily volume


[PASS] Every auction row matches a CRSP stock-day — 0 unmatched


[PASS] Options (date, permno) unique
[PASS] Options volume / open interest non-negative
[PASS] Open interest expiring within 7 days never exceeds total


## 2. Calendar, reference lists and market series

In [4]:
rule_days = TRADING_DAYS[(TRADING_DAYS >= data_days[0]) & (TRADING_DAYS <= data_days[-1])]
check("Rule-based NYSE calendar equals the trading dates in CRSP", rule_days.equals(data_days), f"{len(data_days):,} trading days")
check("FOMC list: 8 scheduled meetings per year", (fomc[fomc.meeting_type == "scheduled"].groupby(fomc.meeting_start_date.dt.year).size() == 8).all())
check("FOMC announcement dates are trading days", fomc_dates.isin(TRADING_DAYS).all())
check("MSCI list: 4 reviews per year, in Feb/May/Aug/Nov", (msci_dates.groupby(msci_dates.dt.year).size() == 4).all() and msci_dates.dt.month.isin([2, 5, 8, 11]).all())

in_data = calendar[calendar.date.isin(data_days)]
for flag, etype in [("sp_quarterly_rebal", "sp_quarterly_rebalance"), ("russell_recon", "russell_reconstitution")]:
    listed = {d for d in index_events.loc[index_events.event_type == etype, "date"] if data_days[0] <= d <= data_days[-1]}
    check(f"Rule-based {flag} dates equal the published list", set(in_data.loc[in_data[flag] == 1, "date"]) == listed, f"{len(listed)} dates")

repo_calendar = DATA_ROOT / "events" / "event_calendar.parquet"
if repo_calendar.exists():
    m = in_data.merge(read_file(repo_calendar), on="date", suffixes=("", "_repo"))
    monthly_opex = m["monthly_opex_ex_quarterly"] + m["sp_quarterly_rebal"]
    pairs = [(m["is_early_close"], "is_early_close_repo"), (monthly_opex, "monthly_options_expiry"), (m["sp_quarterly_rebal"], "quarterly_options_expiry"),
             (m["is_last_trading_day_month"], "is_last_trading_day_month_repo"), (m["is_last_trading_day_quarter"], "is_last_trading_day_quarter_repo"),
             (m["is_last_trading_day_year"], "is_last_trading_day_year_repo"), (m["is_first_trading_day_month"], "is_first_trading_day_month_repo")]
    for ours, theirs in pairs:
        diff = int((ours.astype(int) != m[theirs].astype(int)).sum())
        check(f"Calendar flag equals the repo event calendar: {theirs.replace('_repo', '')}", diff == 0, f"{diff} differing days of {len(m):,}")

repo_market = DATA_ROOT / "market" / "market_daily.parquet"
if repo_market.exists():
    j = market_series(crsp_raw).join(read_file(repo_market).set_index("date")[["market_return_vw", "market_total_volume"]], rsuffix="_repo").dropna()
    check("Recomputed market return equals the repo's market_daily", np.allclose(j.market_return_vw, j.market_return_vw_repo, atol=1e-9))
    check("Recomputed market volume equals the repo's market_daily", np.allclose(j.market_total_volume, j.market_total_volume_repo, rtol=1e-9))

[PASS] Rule-based NYSE calendar equals the trading dates in CRSP — 3,772 trading days
[PASS] FOMC list: 8 scheduled meetings per year
[PASS] FOMC announcement dates are trading days
[PASS] MSCI list: 4 reviews per year, in Feb/May/Aug/Nov
[PASS] Rule-based sp_quarterly_rebal dates equal the published list — 60 dates
[PASS] Rule-based russell_recon dates equal the published list — 15 dates
[PASS] Calendar flag equals the repo event calendar: is_early_close — 0 differing days of 3,772
[PASS] Calendar flag equals the repo event calendar: monthly_options_expiry — 0 differing days of 3,772
[PASS] Calendar flag equals the repo event calendar: quarterly_options_expiry — 0 differing days of 3,772
[PASS] Calendar flag equals the repo event calendar: is_last_trading_day_month — 0 differing days of 3,772
[PASS] Calendar flag equals the repo event calendar: is_last_trading_day_quarter — 0 differing days of 3,772
[PASS] Calendar flag equals the repo event calendar: is_last_trading_day_year — 0 diff

[PASS] Recomputed market return equals the repo's market_daily
[PASS] Recomputed market volume equals the repo's market_daily


## 3. Look-ahead test

Take a forecast day *T*. Rebuild every feature twice from the two years of raw data ending at *T*: once as is, and once after deleting everything only known
at or after the close of *T* (that day's prices, volumes, returns, market caps, auction prints, same-day options fields). If any feature for day *T* changes, it was using day-*T* information.

In [5]:
def lookahead_test(T):
    T = pd.Timestamp(T); start = T - pd.Timedelta(days=760)
    c = crsp_raw[(crsp_raw.date >= start) & (crsp_raw.date <= T)].copy()
    a = auction_raw[(auction_raw.date >= start) & (auction_raw.date <= T)].copy()
    o = options_raw[(options_raw.date >= start) & (options_raw.date <= T)].copy() if options_raw is not None else None
    cal = calendar[calendar.date >= c.date.min()].reset_index(drop=True)
    with_T = build_features(c, a, o, cal)
    c.loc[c.date == T, ["close", "daily_volume", "daily_price_volume", "return_total", "market_cap_usd"]] = np.nan
    if o is not None:
        o.loc[o.date == T, OPTION_SAME_DAY] = np.nan
    without_T = build_features(c, a[a.date < T], o, cal)
    x = with_T.loc[with_T.date == T].set_index("permno")[FEATURE_COLUMNS].astype("float64")
    y = without_T.loc[without_T.date == T].set_index("permno")[FEATURE_COLUMNS].astype("float64").reindex(x.index)
    same = np.isclose(x, y, equal_nan=True, rtol=1e-10, atol=1e-12).all(axis=0)
    return len(x), [col for col, ok in zip(FEATURE_COLUMNS, same) if not ok]

for T in ["2023-03-17", "2024-06-28", "2025-05-30"]:      # S&P quarterly rebalance, Russell reconstitution, MSCI review
    n_stocks, changed = lookahead_test(T)
    check(f"Look-ahead test on {T}: no feature uses day-T data", len(changed) == 0, f"{n_stocks:,} stocks, {len(FEATURE_COLUMNS)} features; changed: {changed or 'none'}")

[PASS] Look-ahead test on 2023-03-17: no feature uses day-T data — 1,339 stocks, 40 features; changed: none


[PASS] Look-ahead test on 2024-06-28: no feature uses day-T data — 1,278 stocks, 40 features; changed: none


[PASS] Look-ahead test on 2025-05-30: no feature uses day-T data — 1,241 stocks, 40 features; changed: none


## 4. Output table

In [6]:
check("Feature table (date, permno) unique", not features.duplicated(["date", "permno"]).any())
check("Target never missing on model-ready rows", features.loc[features.model_ready, "y_log_ratio"].notna().all())
check("Binary flags only take values 0/1", all(set(features[c].dropna().unique()) <= {0, 1} for c in DATE_FLAGS))

rng = np.random.default_rng(7); bad = 0       # recompute the E0 reference by hand for 1,000 sampled stock-days
for p in rng.choice(features.permno.unique(), 25, replace=False):
    s = full.loc[full.permno == p, ["auction_volume", "median20_auction_volume"]].reset_index(drop=True)
    for i in rng.choice(len(s), min(40, len(s)), replace=False):
        prior = s.auction_volume.iloc[max(0, i - 20):i].dropna()
        bad += not np.isclose(s.median20_auction_volume.iloc[i], np.median(prior) if len(prior) else np.nan, equal_nan=True)
check("median20_auction_volume matches a by-hand recomputation", bad == 0, f"{bad} mismatches in 1,000 sampled stock-days")

ready = features[features.model_ready]
pd.DataFrame({"missing_pct": (ready[FEATURE_COLUMNS].isna().mean() * 100).round(3),
              "mean": ready[FEATURE_COLUMNS].astype("float64").mean(),
              "min": ready[FEATURE_COLUMNS].astype("float64").min(), "max": ready[FEATURE_COLUMNS].astype("float64").max()})

[PASS] Feature table (date, permno) unique
[PASS] Target never missing on model-ready rows
[PASS] Binary flags only take values 0/1


[PASS] median20_auction_volume matches a by-hand recomputation — 0 mismatches in 1,000 sampled stock-days


,missing_pct,mean,min,max
median20_auction_volume,0.0000,"142,198.8423",1.0000,"9,465,336.5000"
log_median20_auction_volume,0.0000,10.7407,0.6931,16.0631
n_obs_median20,0.0000,19.9128,10.0000,20.0000
sp_quarterly_rebal,0.0000,0.0160,0.0000,1.0000
russell_recon,0.0000,0.0040,0.0000,1.0000
msci_review,0.0000,0.0160,0.0000,1.0000
any_index_event,0.0000,0.0360,0.0000,1.0000
monthly_opex_ex_quarterly,0.0000,0.0320,0.0000,1.0000
is_last_trading_day_month,0.0000,0.0480,0.0000,1.0000
is_last_trading_day_quarter,0.0000,0.0160,0.0000,1.0000


## 5. Live row — features for the next trading day

The next trading day after the data ends is appended with every same-day value missing, and the same code is run. This is what a live forecast would use.
Open-interest features are missing here because they come from the OCC's morning file in live use.

In [7]:
last_day = data_days[-1]
live_day = TRADING_DAYS[TRADING_DAYS > last_day][0]
recent = crsp_raw[crsp_raw.date >= last_day - pd.Timedelta(days=760)]
live_crsp = pd.concat([recent, recent.loc[recent.date == last_day, ["permno", "ticker"]].assign(date=live_day)], ignore_index=True)
live_options = options_raw[options_raw.date >= recent.date.min()] if options_raw is not None else None
live = build_features(live_crsp, auction_raw[auction_raw.date >= recent.date.min()], live_options,
                      calendar[calendar.date >= recent.date.min()].reset_index(drop=True))
live = live.loc[live.date == live_day, ID_COLUMNS + FEATURE_COLUMNS].reset_index(drop=True)
live.to_parquet(OUTPUT_DIR / f"live_features_{live_day:%Y%m%d}.parquet", index=False)

for name, covered, needed in [("fomc", fomc["announcement_date"].max(), live_day), ("msci", msci_dates.max(), live_day),
                              ("sp500", sp_spells["member_to"].max(), last_day)]:
    check(f"Reference list '{name}' covers the live date {live_day:%Y-%m-%d}", covered >= needed,
          f"list ends {covered:%Y-%m-%d}" + ("; extend it before relying on this flag in live use" if covered < needed else ""), warn_only=True)
print(f"Live feature row for {live_day:%Y-%m-%d}: {len(live):,} stocks")
if options_raw is not None and options_raw.date.max() < last_day:
    print(f"Note: the options layer ends {options_raw.date.max():%Y-%m-%d}, so options features are missing after that date.")
pd.DataFrame({"available_pct": (live[FEATURE_COLUMNS].notna().mean() * 100).round(1)})

[PASS] Reference list 'fomc' covers the live date 2026-01-02 — list ends 2026-12-09
[WARN] Reference list 'msci' covers the live date 2026-01-02 — list ends 2025-11-24; extend it before relying on this flag in live use
[PASS] Reference list 'sp500' covers the live date 2026-01-02 — list ends 2025-12-31
Live feature row for 2026-01-02: 1,234 stocks
Note: the options layer ends 2025-08-29, so options features are missing after that date.


,available_pct
median20_auction_volume,99.8000
log_median20_auction_volume,99.8000
n_obs_median20,100.0000
sp_quarterly_rebal,100.0000
russell_recon,100.0000
msci_review,100.0000
any_index_event,100.0000
monthly_opex_ex_quarterly,100.0000
is_last_trading_day_month,100.0000
is_last_trading_day_quarter,100.0000


## 6. E0 → E5 results compared with the midterm slide

Ordinary least squares on `y_log_ratio`, expanding window: for each test year 2020–2025, fit on all model-ready rows from 2015 up to the previous year-end.
Missing values are filled with the training median and continuous features are capped at the training 0.5th / 99.5th percentiles (both learned in the training window only).
Metric: pooled out-of-sample log RMSE. E5 here also includes stock-level volatility, volume and size, so its gain is larger than on the slide.

In [8]:
LOG_FEATURES = ["volatility20", "abnormal_volume_lag1", "market_volatility20", "market_abnormal_volume_lag1"]
LADDER = [
    ("E0  historical reference + constant", []),
    ("E1  + index rebalancing events", ["sp_quarterly_rebal", "russell_recon", "msci_review"] + ([c for c in SP_FEATURES if c != "td_to_sp500_change"] if USE_SP500_STOCK_LEVEL else [])),
    ("E2  + monthly options expiration", ["monthly_opex_ex_quarterly"]),
    ("E3  + institutional calendar", ["is_last_trading_day_month", "is_last_trading_day_quarter", "is_last_trading_day_year", "is_first_trading_day_month", "is_early_close"]),
    ("E4  + FOMC meeting dates", ["fomc_announcement_day"]),
    ("E5  + volatility / volume regimes and interactions", E5_FEATURES),
] + ([("E5+ + stock-level options exposure (through Aug 2025)", OPT_FEATURES)] if USE_OPTIONS else [])

def fit_predict(train, test, cols):
    if not cols:
        return np.full(len(test), train["y_log_ratio"].mean())
    Xtr, Xte = train[cols].astype("float64").copy(), test[cols].astype("float64").copy()
    for c in cols:
        if c in LOG_FEATURES:
            Xtr[c], Xte[c] = np.log(Xtr[c].where(Xtr[c] > 0)), np.log(Xte[c].where(Xte[c] > 0))
        if Xtr[c].nunique() > 2:
            lo, hi = Xtr[c].quantile([0.005, 0.995]); Xtr[c], Xte[c] = Xtr[c].clip(lo, hi), Xte[c].clip(lo, hi)
        fill = Xtr[c].median(); fill = 0.0 if np.isnan(fill) else fill
        Xtr[c], Xte[c] = Xtr[c].fillna(fill), Xte[c].fillna(fill)
    A = np.column_stack([np.ones(len(Xtr)), Xtr.to_numpy()])
    beta, *_ = np.linalg.lstsq(A.T @ A, A.T @ train["y_log_ratio"].to_numpy(), rcond=None)
    return np.column_stack([np.ones(len(Xte)), Xte.to_numpy()]) @ beta

ready = features[features.model_ready].assign(year=lambda d: d.date.dt.year)
test_years = list(range(2020, int(ready.year.max()) + 1))
rows, cols = [], []
for name, add in LADDER:
    cols = cols + add
    err = np.concatenate([ready.loc[ready.year == y, "y_log_ratio"].to_numpy() - fit_predict(ready[ready.year < y], ready[ready.year == y], cols) for y in test_years])
    rows.append({"specification": name, "oos_log_rmse": np.sqrt(np.mean(err ** 2))})
ladder = pd.DataFrame(rows)
slide = [0.6164, 0.540, 0.539, 0.529, 0.528, 0.5250]
ladder["midterm_slide"] = slide + [np.nan] * (len(ladder) - len(slide))
ladder["change_vs_previous"] = ladder["oos_log_rmse"].diff()
ladder.to_csv(OUTPUT_DIR / "sanity_model_ladder.csv", index=False)
print(f"Out-of-sample stock-days ({test_years[0]}-{test_years[-1]}): {int(ready.year.isin(test_years).sum()):,}")
check("E0 matches the midterm slide (0.6164) within 0.002", abs(ladder.oos_log_rmse.iloc[0] - 0.6164) < 0.002, f"{ladder.oos_log_rmse.iloc[0]:.4f}")
ladder

Out-of-sample stock-days (2020-2025): 1,882,373
[PASS] E0 matches the midterm slide (0.6164) within 0.002 — 0.6163


,specification,oos_log_rmse,midterm_slide,change_vs_previous
0,E0 historical reference + constant,0.6163,0.6164,NaN
1,E1 + index rebalancing events,0.5404,0.5400,-0.0759
2,E2 + monthly options expiration,0.5400,0.5390,-0.0004
3,E3 + institutional calendar,0.5307,0.5290,-0.0093
4,E4 + FOMC meeting dates,0.5303,0.5280,-0.0004
5,E5 + volatility / volume regimes and interact...,0.5142,0.5250,-0.0161
6,E5+ + stock-level options exposure (through Au...,0.5141,NaN,-0.0002


## 7. Summary of checks

In [9]:
qc = pd.DataFrame(QC)
qc.to_csv(OUTPUT_DIR / "qc_report.csv", index=False)
print(qc.status.value_counts().to_dict())
qc

{'PASS': 35, 'WARN': 1}


,check,status,detail
0,"CRSP (date, permno) unique",PASS,
1,"Auction (date, permno) unique",PASS,
2,CRSP daily volume non-negative,PASS,
3,Auction volume positive where present,PASS,
4,target_found agrees with auction_volume presence,PASS,
5,Auction volume never exceeds CRSP daily volume,PASS,
6,Every auction row matches a CRSP stock-day,PASS,0 unmatched
7,"Options (date, permno) unique",PASS,
8,Options volume / open interest non-negative,PASS,
9,Open interest expiring within 7 days never exc...,PASS,
